In [ ]:
import os, glob, random, copy, warnings, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from scipy.optimize import linprog
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, roc_auc_score

print('Imports done.')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    torch.backends.cudnn.benchmark = True
    print(f'Using GPU: {torch.cuda.get_device_name(0)}')
else:
    print('No GPU detected — running on CPU.')

In [ ]:
dataset_names = ['Compas','Compas','Bank','Default','Adult','Adult']
protected_attributes = ['sex','race','age','sex','sex','race']

CSV_DIR = r'/home/shivampathak/Desktop/New Folder/preprocessed_data'         
RESULTS_DIR = r'C/home/shivampathak/Desktop/New Folder/results_aganosticfair'  
TARGET_COL_DEFAULT = 'Probability'
TARGET_COL_OVERRIDES = {}

METHOD_NAME = 'AgnosticFair'

os.makedirs(RESULTS_DIR, exist_ok=True)
MASTER_LOG_PATH   = os.path.join(RESULTS_DIR, 'master_log.csv')
HISTORY_LOG_PATH  = os.path.join(RESULTS_DIR, 'round_history_log.csv')
ROC_DIR           = os.path.join(RESULTS_DIR, 'roc_data')
os.makedirs(ROC_DIR, exist_ok=True)


NUM_CLIENTS  = 10
LOCAL_EPOCHS = 5        
BATCH_SIZE   = 32
LR           = 0.01     
TEST_SIZE    = 0.20

NUM_ROUNDS_MAIN = 30
SEEDS_MAIN = [42, 43, 44, 45, 46]


NUM_BASES    = 200   
KERNEL_SIGMA = 1.0   
ALPHA_UPPER_B = 5.0    
FAIRNESS_TAU  = 0.05   
FAIR_LAMBDA   = 2.0    
LAMBDA_GRID = [0.0, 0.5, 1.0, 2.0, 5.0, 10.0]   
SEEDS_SENSITIVITY = [42, 43]

SAVE_MODEL_WEIGHTS = False
SAVE_ROC_DATA       = True

print('Config set.')
print(f'{len(dataset_names)} dataset-attribute combinations queued for method={METHOD_NAME}.')

In [ ]:
class LogisticRegressionModel(nn.Module):
    """f(x) = sigmoid(w.x + b) — the paper's own model choice (Section 4.1: "we use the logistic
    regression model to evaluate our proposed algorithm")."""
    def __init__(self, input_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)
    def forward(self, x):
        return torch.sigmoid(self.linear(x))
    def logits(self, x):
        return self.linear(x)
    def decision_distance(self, x):
        """Signed distance to the decision boundary, d_w(x) = (w.x+b) / ||w||_2 — the standard
        geometric distance for a linear classifier; the paper does not spell out the exact
        formula for d_w(x), only that it is "the distance to the decision boundary"."""
        z = self.linear(x).squeeze(-1)
        w_norm = self.linear.weight.norm(p=2) + 1e-8
        return z / w_norm


def gaussian_kernel_features(X, bases, sigma):
    """K_m(x) = exp(-||b_m - x||^2 / (2*sigma^2)) for m=1..M (eq. 3.8). Returns (n, M)."""
    d2 = torch.cdist(X, bases, p=2) ** 2
    return torch.exp(-d2 / (2 * sigma ** 2))


def fairness_metrics(y_true, y_pred, protected, y_score=None):
    """DI, SPD, EOD, AOD, FPRD, TNRD, PPD + per-group rates/precision/F1 + AUC.
    signed DI = 1 - p_unpriv/p_priv (ideal = 0)."""
    y_true = np.array(y_true); y_pred = np.array(y_pred); protected = np.array(protected)
    priv_idx = protected == 1
    unpriv_idx = protected == 0
    p_priv   = np.mean(y_pred[priv_idx] == 1)   if priv_idx.sum()   > 0 else 0.0
    p_unpriv = np.mean(y_pred[unpriv_idx] == 1) if unpriv_idx.sum() > 0 else 0.0
    SPD = p_unpriv - p_priv
    DI  = 1.0 - (p_unpriv / p_priv) if p_priv > 1e-9 else np.nan

    def rates(yt, yp):
        if len(yt) == 0:
            return dict(tpr=0.0, fpr=0.0, tnr=0.0, fnr=0.0, precision=0.0, f1=0.0, acc=0.0)
        tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
        tnr = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        f1 = 2 * precision * tpr / (precision + tpr) if (precision + tpr) > 0 else 0.0
        acc_g = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0.0
        return dict(tpr=tpr, fpr=fpr, tnr=tnr, fnr=fnr, precision=precision, f1=f1, acc=acc_g)

    r_priv    = rates(y_true[priv_idx],   y_pred[priv_idx])
    r_unpriv  = rates(y_true[unpriv_idx], y_pred[unpriv_idx])
    r_overall = rates(y_true, y_pred)

    EOD  = r_unpriv['tpr'] - r_priv['tpr']
    AOD  = ((r_unpriv['fpr'] - r_priv['fpr']) + (r_unpriv['tpr'] - r_priv['tpr'])) / 2
    FPRD = r_unpriv['fpr'] - r_priv['fpr']
    TNRD = r_unpriv['tnr'] - r_priv['tnr']
    PPD  = r_unpriv['precision'] - r_priv['precision']

    out = {
        'DI': DI, 'SPD': SPD, 'EOD': EOD, 'AOD': AOD, 'FPRD': FPRD, 'TNRD': TNRD, 'PPD': PPD,
        'tpr_priv': r_priv['tpr'],   'tpr_unpriv': r_unpriv['tpr'],
        'fpr_priv': r_priv['fpr'],   'fpr_unpriv': r_unpriv['fpr'],
        'tnr_priv': r_priv['tnr'],   'tnr_unpriv': r_unpriv['tnr'],
        'fnr_priv': r_priv['fnr'],   'fnr_unpriv': r_unpriv['fnr'],
        'precision_priv': r_priv['precision'], 'precision_unpriv': r_unpriv['precision'],
        'f1_priv': r_priv['f1'],     'f1_unpriv': r_unpriv['f1'],
        'f1_overall': r_overall['f1'], 'precision_overall': r_overall['precision'],
    }
    if y_score is not None:
        y_score = np.array(y_score)
        try:
            out['auc_overall'] = roc_auc_score(y_true, y_score)
        except ValueError:
            out['auc_overall'] = np.nan
        try:
            out['auc_priv'] = (roc_auc_score(y_true[priv_idx], y_score[priv_idx])
                                if priv_idx.sum() > 0 and len(np.unique(y_true[priv_idx])) > 1 else np.nan)
        except ValueError:
            out['auc_priv'] = np.nan
        try:
            out['auc_unpriv'] = (roc_auc_score(y_true[unpriv_idx], y_score[unpriv_idx])
                                  if unpriv_idx.sum() > 0 and len(np.unique(y_true[unpriv_idx])) > 1 else np.nan)
        except ValueError:
            out['auc_unpriv'] = np.nan
    return out


def full_eval(model, X_tensor, y_np, prot_np, return_scores=False):
    model.eval()
    with torch.no_grad():
        probs = model(X_tensor).squeeze().cpu().numpy()
    preds = (probs >= 0.5).astype(float)
    acc  = accuracy_score(y_np, preds)
    bacc = balanced_accuracy_score(y_np, preds)
    fm   = fairness_metrics(y_np.astype(int), preds.astype(int), prot_np, y_score=probs)
    if return_scores:
        return acc, bacc, fm, probs
    return acc, bacc, fm

print('Model / metric definitions ready.')

In [ ]:
def solve_alpha_lp(c, e, d, tau, alpha_upper_b):
    """Solves eq. (3.19): maximize c.alpha s.t. e.alpha = 1, 0<=alpha<=B, |d.alpha|<=tau.
    scipy.optimize.linprog only minimizes, so we minimize -c.alpha."""
    M = len(c)
    A_eq = e.reshape(1, -1)
    b_eq = np.array([1.0])
    A_ub = np.vstack([d.reshape(1, -1), -d.reshape(1, -1)])
    b_ub = np.array([tau, tau])
    bounds = [(0.0, alpha_upper_b)] * M
    res = linprog(c=-c, A_eq=A_eq, b_eq=b_eq, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')
    if res.success:
        return res.x
    return None   


def run_agnosticfair_pipeline(df, protected_attr, target_col,
                               num_clients=NUM_CLIENTS, num_rounds=NUM_ROUNDS_MAIN,
                               local_epochs=LOCAL_EPOCHS, batch_size=BATCH_SIZE, lr=LR,
                               fair_lambda=FAIR_LAMBDA, tau=FAIRNESS_TAU,
                               num_bases=NUM_BASES, sigma=KERNEL_SIGMA, alpha_upper_b=ALPHA_UPPER_B,
                               seed=42):
    
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    if DEVICE.type == 'cuda': torch.cuda.manual_seed_all(seed)

    train_df, test_df = train_test_split(df, test_size=TEST_SIZE, random_state=seed, shuffle=True)
    train_df = train_df.sample(frac=1, random_state=seed).reset_index(drop=True)
    client_idx_splits = np.array_split(np.arange(len(train_df)), num_clients)
    clients_raw = [train_df.iloc[idx].reset_index(drop=True) for idx in client_idx_splits]

    X_test_np = test_df.drop(columns=[target_col]).values.astype(np.float32)
    y_test_np = test_df[target_col].values.astype(np.float32)
    prot_test = test_df[protected_attr].values
    X_test_tensor = torch.tensor(X_test_np).to(DEVICE)
    input_dim = X_test_np.shape[1]

    s_bar_global = float(train_df[protected_attr].mean())   
    n_total = len(train_df)

    
    rng = np.random.RandomState(seed)
    bases_np = rng.uniform(0.0, 1.0, size=(num_bases, input_dim)).astype(np.float32)
    bases = torch.tensor(bases_np).to(DEVICE)

    global_model = LogisticRegressionModel(input_dim).to(DEVICE)
    alpha_t = np.full(num_bases, 1.0 / num_bases, dtype=np.float64)   
    others_frozen = {cid: 0.0 for cid in range(num_clients)}          
    bce_none = nn.BCELoss(reduction='none')

    round_history = []
    g_acc = g_bacc = None
    g_fm = None
    g_probs = None

    for rnd in range(1, num_rounds + 1):
        local_states = []
        client_n = []
        agg_c = np.zeros(num_bases); agg_e = np.zeros(num_bases); agg_d = np.zeros(num_bases)
        cd_k_scalars = {}

        alpha_tensor = torch.tensor(alpha_t, dtype=torch.float32, device=DEVICE)

        for cid in range(num_clients):
            model_k = LogisticRegressionModel(input_dim).to(DEVICE)
            model_k.load_state_dict(global_model.state_dict())
            model_k.train()
            opt = optim.Adam(model_k.parameters(), lr=lr)

            cdf = clients_raw[cid]
            X_c = torch.tensor(cdf.drop(columns=[target_col]).values.astype(np.float32))
            y_c = torch.tensor(cdf[target_col].values.astype(np.float32))
            s_c = torch.tensor(cdf[protected_attr].values.astype(np.float32))
            loader = DataLoader(TensorDataset(X_c, y_c, s_c), batch_size=batch_size, shuffle=True, drop_last=False)
            n_k = len(cdf)

            cd_others = others_frozen[cid]

            
            for _ in range(local_epochs):
                for bX, by, bs in loader:
                    if bX.size(0) < 1:
                        continue
                    bX, by, bs = bX.to(DEVICE), by.to(DEVICE), bs.to(DEVICE)

                    K_batch = gaussian_kernel_features(bX, bases, sigma)          
                    theta = (K_batch @ alpha_tensor)                              

                    probs = model_k(bX).squeeze(-1)
                    per_sample_loss = bce_none(probs, by)                         
                    weighted_loss = (theta * per_sample_loss).mean()              

                    d_w = model_k.decision_distance(bX)                          
                    s_centered = bs - s_bar_global
                    cd_live = (s_centered * theta * d_w).mean()                  
                    cd_estimate = cd_others + cd_live
                    fairness_pen = (cd_estimate - tau) ** 2                      

                    total_loss = weighted_loss + fair_lambda * fairness_pen

                    opt.zero_grad()
                    total_loss.backward()
                    opt.step()

            
            model_k.eval()
            with torch.no_grad():
                X_full = X_c.to(DEVICE)
                y_full = y_c.to(DEVICE)
                s_full = s_c.to(DEVICE)
                K_full = gaussian_kernel_features(X_full, bases, sigma)                    
                probs_full = model_k(X_full).squeeze(-1)
                l_full = bce_none(probs_full, y_full)                                      
                d_full = model_k.decision_distance(X_full)                                 
                s_centered_full = s_full - s_bar_global

                c_k = (K_full * l_full.unsqueeze(1)).sum(dim=0).cpu().numpy()             
                e_k = K_full.sum(dim=0).cpu().numpy()                                      
                d_k = (K_full * (s_centered_full * d_full).unsqueeze(1)).sum(dim=0).cpu().numpy()  

            agg_c += c_k; agg_e += e_k; agg_d += d_k
            cd_k_scalars[cid] = float(np.dot(alpha_t, d_k) / n_total)   

            client_n.append(n_k)
            local_states.append(copy.deepcopy(model_k.state_dict()))

        
        p = num_clients
        def agg_state_uniform(states):
            out = {}
            for key in states[0]:
                out[key] = sum(states[i][key].float() for i in range(p)) / p
            return out
        global_model.load_state_dict(agg_state_uniform(local_states))

        
        c_agg = agg_c / n_total
        e_agg = agg_e / n_total
        d_agg = agg_d / n_total
        new_alpha = solve_alpha_lp(c_agg, e_agg, d_agg, tau, alpha_upper_b)
        if new_alpha is not None:
            alpha_t = new_alpha
        
        cd_total_this_round = sum(cd_k_scalars.values())
        others_frozen = {cid: cd_total_this_round - cd_k_scalars[cid] for cid in range(num_clients)}

        
        g_acc, g_bacc, g_fm, g_probs = full_eval(global_model, X_test_tensor, y_test_np, prot_test, return_scores=True)
        g_di = g_fm['DI'] if not np.isnan(g_fm['DI']) else 0.0

        round_history.append({
            'round': rnd,
            'global_acc': g_acc, 'global_bacc': g_bacc, 'global_di1': 1 - g_di,
            'global_spd': g_fm['SPD'], 'global_eod': g_fm['EOD'], 'global_aod': g_fm['AOD'],
            'global_auc': g_fm.get('auc_overall', np.nan),
        })

    
    result = {
        'selected_round':  num_rounds,
        'selected_type':   'global',
        'final_acc':       g_acc,
        'final_bacc':      g_bacc,
        'final_di1':       1 - (g_fm['DI'] if not np.isnan(g_fm['DI']) else 0.0),
        'final_spd':       g_fm['SPD'],
        'final_eod':       g_fm['EOD'],
        'final_aod':       g_fm['AOD'],
        'final_fprd': g_fm['FPRD'], 'final_tnrd': g_fm['TNRD'], 'final_ppd': g_fm['PPD'],
        'tpr_priv': g_fm['tpr_priv'], 'tpr_unpriv': g_fm['tpr_unpriv'],
        'fpr_priv': g_fm['fpr_priv'], 'fpr_unpriv': g_fm['fpr_unpriv'],
        'tnr_priv': g_fm['tnr_priv'], 'tnr_unpriv': g_fm['tnr_unpriv'],
        'fnr_priv': g_fm['fnr_priv'], 'fnr_unpriv': g_fm['fnr_unpriv'],
        'precision_priv': g_fm['precision_priv'], 'precision_unpriv': g_fm['precision_unpriv'],
        'f1_priv': g_fm['f1_priv'], 'f1_unpriv': g_fm['f1_unpriv'],
        'f1_overall': g_fm['f1_overall'], 'precision_overall': g_fm['precision_overall'],
        'auc_overall': g_fm.get('auc_overall', np.nan),
        'auc_priv': g_fm.get('auc_priv', np.nan),
        'auc_unpriv': g_fm.get('auc_unpriv', np.nan),
        'final_probs':     g_probs,
        'y_test':          y_test_np,
        'prot_test':       prot_test,
        'round_history':   round_history,
    }
    if SAVE_MODEL_WEIGHTS:
        result['state_dict'] = global_model.state_dict()
    return result

print('run_agnosticfair_pipeline() ready.')

In [ ]:
def get_csv_path(dataset_name):
    return os.path.join(CSV_DIR, f'{dataset_name.lower()}_processed.csv')


def load_dataset(dataset_name, protected_attr):
    path = get_csv_path(dataset_name)
    df = pd.read_csv(path)
    target_col = TARGET_COL_OVERRIDES.get(dataset_name, TARGET_COL_DEFAULT)
    feature_cols = [c for c in df.columns if c not in [target_col, protected_attr]]
    numeric_cols = df[feature_cols].select_dtypes(include=['int64', 'float64']).columns
    scaler = MinMaxScaler()
    df = df.copy()
    df[numeric_cols] = scaler.fit_transform(df[numeric_cols])
    df[target_col] = df[target_col].astype(int)
    df[protected_attr] = df[protected_attr].astype(int)
    return df, target_col


def _norm(v):
    return '' if v is None else str(v)


def load_completed_keys():
    if not os.path.exists(MASTER_LOG_PATH):
        return set()
    log = pd.read_csv(MASTER_LOG_PATH)
    keys = set(zip(
        log['dataset'], log['protected_attr'], log['experiment_type'], log['method'], log['variant'],
        log['param_name'].fillna('').astype(str), log['param_value'].fillna('').astype(str),
        log['seed'].astype(str)
    ))
    return keys


def append_master_row(row: dict):
    df_row = pd.DataFrame([row])
    header = not os.path.exists(MASTER_LOG_PATH)
    df_row.to_csv(MASTER_LOG_PATH, mode='a', header=header, index=False)


def append_history_rows(dataset, protected_attr, experiment_type, method, variant,
                         param_name, param_value, seed, round_history):
    rows = []
    for rh in round_history:
        r = dict(rh)
        r.update({
            'dataset': dataset, 'protected_attr': protected_attr,
            'experiment_type': experiment_type, 'method': method, 'variant': variant,
            'param_name': _norm(param_name), 'param_value': _norm(param_value),
            'seed': seed,
        })
        rows.append(r)
    df_rows = pd.DataFrame(rows)
    header = not os.path.exists(HISTORY_LOG_PATH)
    df_rows.to_csv(HISTORY_LOG_PATH, mode='a', header=header, index=False)


def save_roc_data(method, dataset_label, protected_attr, seed, y_test, final_probs, prot_test,
                   fair_lambda=None, tau=None, selected_round=None, selected_type=None,
                   experiment_type='main', param_name=None, param_value=None):
    tag = '' if experiment_type == 'main' else f'__{experiment_type}-{_norm(param_name)}{_norm(param_value)}'
    fname = f'{method}__{dataset_label}{tag}__seed{seed}.npz'
    np.savez_compressed(os.path.join(ROC_DIR, fname),
                         y_true=np.asarray(y_test), y_score=np.asarray(final_probs),
                         protected=np.asarray(prot_test),
                         method=method, fair_lambda=fair_lambda, tau=tau,
                         num_clients=NUM_CLIENTS, local_epochs=LOCAL_EPOCHS, batch_size=BATCH_SIZE,
                         num_bases=NUM_BASES, sigma=KERNEL_SIGMA, alpha_upper_b=ALPHA_UPPER_B,
                         selected_round=selected_round, selected_type=selected_type)

print('Logging helpers ready.')

In [ ]:
completed = load_completed_keys()

def already_done(key):
    return key in completed

def mark_done(key):
    completed.add(key)

def make_key(dataset, protected_attr, experiment_type, method, variant, param_name, param_value, seed):
    return (dataset, protected_attr, experiment_type, method, variant,
            _norm(param_name), _norm(param_value), str(seed))


def _extended_row_fields(res):
    return {
        'fprd': res['final_fprd'], 'tnrd': res['final_tnrd'], 'ppd': res['final_ppd'],
        'tpr_priv': res['tpr_priv'], 'tpr_unpriv': res['tpr_unpriv'],
        'fpr_priv': res['fpr_priv'], 'fpr_unpriv': res['fpr_unpriv'],
        'tnr_priv': res['tnr_priv'], 'tnr_unpriv': res['tnr_unpriv'],
        'fnr_priv': res['fnr_priv'], 'fnr_unpriv': res['fnr_unpriv'],
        'precision_priv': res['precision_priv'], 'precision_unpriv': res['precision_unpriv'],
        'f1_priv': res['f1_priv'], 'f1_unpriv': res['f1_unpriv'],
        'f1_overall': res['f1_overall'], 'precision_overall': res['precision_overall'],
        'auc_overall': res['auc_overall'], 'auc_priv': res['auc_priv'], 'auc_unpriv': res['auc_unpriv'],
    }


for ds_name, prot_attr in zip(dataset_names, protected_attributes):
    ds_label = f'{ds_name}_{prot_attr}'
    print(f'\n=== {ds_label}  [{METHOD_NAME}] ===')

    try:
        df, target_col = load_dataset(ds_name, prot_attr)
    except FileNotFoundError:
        print(f'  !! CSV not found for {ds_name} at {get_csv_path(ds_name)} — skipping.')
        continue

    for seed in SEEDS_MAIN:
        key = make_key(ds_label, prot_attr, 'main', METHOD_NAME, 'full', None, None, seed)
        if already_done(key):
            continue
        res = run_agnosticfair_pipeline(df, prot_attr, target_col, num_rounds=NUM_ROUNDS_MAIN, seed=seed)
        row = {
            'dataset': ds_label, 'protected_attr': prot_attr,
            'experiment_type': 'main', 'method': METHOD_NAME, 'variant': 'full',
            'param_name': '', 'param_value': '', 'seed': seed,
            'selected_round': res['selected_round'], 'selected_type': res['selected_type'],
            'acc': res['final_acc'], 'bacc': res['final_bacc'], 'di1': res['final_di1'],
            'spd': res['final_spd'], 'eod': res['final_eod'], 'aod': res['final_aod'],
        }
        row.update(_extended_row_fields(res))
        append_master_row(row)
        append_history_rows(ds_label, prot_attr, 'main', METHOD_NAME, 'full', None, None, seed, res['round_history'])
        if SAVE_ROC_DATA:
            save_roc_data(METHOD_NAME, ds_label, prot_attr, seed, res['y_test'], res['final_probs'], res['prot_test'],
                          fair_lambda=FAIR_LAMBDA, tau=FAIRNESS_TAU,
                          selected_round=res['selected_round'], selected_type=res['selected_type'],
                          experiment_type='main')
        mark_done(key)
        print(f'  [main] seed={seed}  Acc={res["final_acc"]:.4f}  1-DI={res["final_di1"]:.4f}  AUC={res["auc_overall"]:.4f}')

print('\nAll queued AgnosticFair main-experiment runs complete (or already were, and were skipped).')

In [ ]:

for ds_name, prot_attr in zip(dataset_names, protected_attributes):
    ds_label = f'{ds_name}_{prot_attr}'
    try:
        df, target_col = load_dataset(ds_name, prot_attr)
    except FileNotFoundError:
        print(f'  !! CSV not found for {ds_name} — skipping sensitivity sweep.')
        continue

    for lam in LAMBDA_GRID:
        for seed in SEEDS_SENSITIVITY:
            key = make_key(ds_label, prot_attr, 'sensitivity', METHOD_NAME, 'full', 'fair_lambda', lam, seed)
            if already_done(key):
                continue
            res = run_agnosticfair_pipeline(df, prot_attr, target_col, num_rounds=NUM_ROUNDS_MAIN, seed=seed,
                                             fair_lambda=lam)
            row = {
                'dataset': ds_label, 'protected_attr': prot_attr,
                'experiment_type': 'sensitivity', 'method': METHOD_NAME, 'variant': 'full',
                'param_name': 'fair_lambda', 'param_value': lam, 'seed': seed,
                'selected_round': res['selected_round'], 'selected_type': res['selected_type'],
                'acc': res['final_acc'], 'bacc': res['final_bacc'], 'di1': res['final_di1'],
                'spd': res['final_spd'], 'eod': res['final_eod'], 'aod': res['final_aod'],
            }
            row.update(_extended_row_fields(res))
            append_master_row(row)
            append_history_rows(ds_label, prot_attr, 'sensitivity', METHOD_NAME, 'full',
                                 'fair_lambda', lam, seed, res['round_history'])
            if SAVE_ROC_DATA:
                save_roc_data(METHOD_NAME, ds_label, prot_attr, seed, res['y_test'], res['final_probs'],
                              res['prot_test'], fair_lambda=lam, tau=FAIRNESS_TAU,
                              selected_round=res['selected_round'], selected_type=res['selected_type'],
                              experiment_type='sensitivity', param_name='fair_lambda', param_value=lam)
            mark_done(key)
            print(f'  [sensitivity] {ds_label}  lambda={lam}  seed={seed}  '
                  f'Acc={res["final_acc"]:.4f}  1-DI={res["final_di1"]:.4f}')

print('\nLambda sensitivity sweep complete (or already was, and was skipped).')